# Distributional RL Research Notebook

## Hypothesis
Test whether a distributional return model can produce better out-of-sample, risk-adjusted position sizing than simple fixed-exposure and point-forecast baselines under realistic frictions.

## Notebook goals
1. Load cached Alpaca bars when available, with a synthetic fallback for reproducibility.
2. Build a research dataset with features, targets, timestamps, and symbol metadata.
3. Run walk-forward evaluation instead of a single train/validation split.
4. Compare `Normal` and `StudentT` distribution families against static and point-forecast baselines.
5. Report performance, turnover, drawdown, and basic distribution-quality diagnostics.
6. Run a small sensitivity analysis over transaction costs and leverage penalties.

## Success criteria
- Higher out-of-sample adjusted Sharpe than the baselines.
- Competitive or better max drawdown.
- Stable behavior across folds, symbols, and simple regime slices.
- Predictive distributions that are at least directionally calibrated.


## 1. Environment and reproducibility

This notebook is designed to run from the repository root. It records the active configuration, package versions, and run timestamp so that each result table is tied to a concrete experiment setup.


In [ ]:
import importlib.util
import subprocess
import sys

for package_spec in ("ngboost>=0.5.0", "matplotlib>=3.7.0"):
    package_name = package_spec.split(">=", 1)[0]
    if importlib.util.find_spec(package_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_spec])


In [ ]:
from __future__ import annotations

import importlib.metadata
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import (
    AlpacaMarketDataStore,
    DistributionalStrategy,
    adjusted_sharpe_ratio,
    build_feature_dataset,
    max_drawdown,
    sharpe_ratio,
    sortino_ratio,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

plt.style.use("seaborn-v0_8-whitegrid")
UTC = timezone.utc


## 2. Research configuration

The notebook now uses a compact but more credible research setup:
- multiple symbols when data is available
- walk-forward folds on unique timestamps
- fixed-exposure baselines plus a simple point-forecast baseline
- distribution diagnostics and regime summaries
- sensitivity analysis on transaction costs and leverage controls


In [ ]:
@dataclass(frozen=True)
class ResearchConfig:
    random_seed: int = 42
    symbols: tuple[str, ...] = ("AAPL", "MSFT", "SPY")
    timeframe: str = "1Day"
    start: datetime = datetime(2022, 1, 1, tzinfo=UTC)
    end: datetime = datetime(2024, 12, 31, tzinfo=UTC)
    storage_root: Path = Path("data/alpaca")

    return_horizon: int = 1
    volatility_window: int = 20
    atr_window: int = 14
    volume_window: int = 20

    min_train_periods: int = 140
    test_periods: int = 40
    step_periods: int = 40
    max_folds: int = 4

    allow_short: bool = False
    baseline_exposures: tuple[float, ...] = (0.0, 1.0, 2.0)
    distribution_families: tuple[str, ...] = ("Normal", "StudentT")
    cost_grid: tuple[float, ...] = (0.0, 0.0005, 0.001)
    leverage_penalty_grid: tuple[float, ...] = (0.0, 0.01, 0.03)
    drawdown_penalty: float = 0.05
    expected_return_weight: float = 1.0
    monte_carlo_samples: int = 400

    model_grid: tuple[tuple[str, int, float], ...] = (
        ("Normal", 20, 0.05),
        ("StudentT", 20, 0.05),
    )

    def simulation_params(self, *, transaction_cost: float = 0.0, leverage_penalty: float = 0.01) -> dict:
        if self.allow_short:
            grid_points = np.linspace(-1.0, 2.0, 13)
        else:
            grid_points = np.linspace(0.0, 2.0, 9)
        return {
            "grid_points": grid_points,
            "n_samples": self.monte_carlo_samples,
            "transaction_cost": transaction_cost,
            "leverage_penalty": leverage_penalty,
            "drawdown_penalty": self.drawdown_penalty,
            "expected_return_weight": self.expected_return_weight,
        }


config = ResearchConfig()
rng = np.random.default_rng(config.random_seed)
np.random.seed(config.random_seed)


## 3. Data loading and validation

Preferred workflow:
1. Load cached parquet bars if they already exist.
2. Download from Alpaca only when credentials are configured and cached data is missing.
3. Fall back to deterministic synthetic data so the notebook remains runnable.
4. Inspect coverage, missingness, and symbol balance before modeling.


In [ ]:
def make_synthetic_bars(symbol: str, periods: int = 420, seed: int = 0) -> pd.DataFrame:
    local_rng = np.random.default_rng(seed)
    timestamps = pd.date_range("2023-01-03 00:00:00+00:00", periods=periods, freq="B")

    drift = {"AAPL": 0.0007, "MSFT": 0.0005, "SPY": 0.0004}.get(symbol, 0.0005)
    vol = {"AAPL": 0.015, "MSFT": 0.013, "SPY": 0.009}.get(symbol, 0.012)
    shocks = local_rng.normal(loc=drift, scale=vol, size=periods)
    close = 100.0 * np.exp(np.cumsum(shocks))
    open_ = close * (1.0 + local_rng.normal(0.0, 0.0025, size=periods))
    high = np.maximum(open_, close) * (1.0 + local_rng.uniform(0.0005, 0.01, size=periods))
    low = np.minimum(open_, close) * (1.0 - local_rng.uniform(0.0005, 0.01, size=periods))
    volume = local_rng.integers(900_000, 3_000_000, size=periods)

    return pd.DataFrame(
        {
            "symbol": symbol,
            "timestamp": timestamps,
            "open": open_,
            "high": high,
            "low": low,
            "close": close,
            "volume": volume,
            "trade_count": np.arange(periods) + 1,
            "vwap": (open_ + high + low + close) / 4.0,
            "timeframe": config.timeframe,
        }
    )


def load_or_create_bars(cfg: ResearchConfig) -> tuple[pd.DataFrame, str]:
    store = AlpacaMarketDataStore(storage_root=cfg.storage_root)
    try:
        bars = store.load_stock_bars(
            symbols=cfg.symbols,
            timeframe=cfg.timeframe,
            start=cfg.start,
            end=cfg.end,
        )
        if bars.empty:
            raise ValueError("No cached bars found for the requested slice.")
        source = "cached_parquet"
    except Exception:
        synthetic_frames = [
            make_synthetic_bars(symbol=symbol, periods=420, seed=cfg.random_seed + idx)
            for idx, symbol in enumerate(cfg.symbols)
        ]
        bars = pd.concat(synthetic_frames, ignore_index=True)
        source = "synthetic_fallback"

    bars = bars.sort_values(["symbol", "timestamp"]).reset_index(drop=True)
    return bars, source


bars, data_source = load_or_create_bars(config)
bars.head()


In [ ]:
data_summary = pd.DataFrame(
    {
        "symbols": [", ".join(sorted(bars["symbol"].unique()))],
        "rows": [len(bars)],
        "timeframe": [bars["timeframe"].mode().iat[0] if not bars.empty else config.timeframe],
        "start": [bars["timestamp"].min()],
        "end": [bars["timestamp"].max()],
        "missing_values": [int(bars.isna().sum().sum())],
        "data_source": [data_source],
    }
)

bars_per_symbol = (
    bars.groupby("symbol")
    .agg(rows=("timestamp", "size"), start=("timestamp", "min"), end=("timestamp", "max"))
    .sort_index()
)

display(data_summary)
display(bars_per_symbol)


## 4. Build the research dataset

The package function `build_feature_dataset` is still the authoritative feature pipeline. For notebook analysis we rebuild the same feature set with timestamp and symbol metadata attached so that fold construction, regime slicing, and failure-case inspection stay aligned with the modeled rows.


In [ ]:
def _last_rank_percentile(window: np.ndarray) -> float:
    last_value = window[-1]
    return float(np.mean(window <= last_value))


def build_research_dataset(
    bars: pd.DataFrame,
    *,
    return_horizon: int,
    volatility_window: int,
    atr_window: int,
    volume_window: int,
) -> pd.DataFrame:
    frame = bars.copy()
    frame["timestamp"] = pd.to_datetime(frame["timestamp"], utc=True)
    frame = frame.sort_values(["symbol", "timestamp"]).reset_index(drop=True)

    grouped = frame.groupby("symbol", group_keys=False)
    prev_close = grouped["close"].shift(1)
    log_close = np.log(frame["close"])
    frame["log_return"] = log_close.groupby(frame["symbol"]).diff()

    intrabar_range = (frame["high"] - frame["low"]).replace(0, np.nan)
    frame["bar_portion"] = ((frame["close"] - frame["low"]) / intrabar_range).clip(0.0, 1.0).fillna(0.5)

    true_range = pd.concat(
        [
            frame["high"] - frame["low"],
            (frame["high"] - prev_close).abs(),
            (frame["low"] - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1)

    frame["realised_vol"] = grouped["log_return"].transform(
        lambda values: values.rolling(window=volatility_window, min_periods=volatility_window).std(ddof=0)
    )
    frame["_true_range"] = true_range
    frame["atr"] = grouped["_true_range"].transform(
        lambda values: values.rolling(window=atr_window, min_periods=atr_window).mean()
    )
    frame["vol_percentile"] = grouped["volume"].transform(
        lambda values: values.rolling(window=volume_window, min_periods=volume_window).apply(_last_rank_percentile, raw=True)
    )
    frame["target_return"] = grouped["close"].transform(
        lambda values: np.log(values.shift(-return_horizon) / values)
    )

    dataset = frame[
        [
            "timestamp",
            "symbol",
            "bar_portion",
            "log_return",
            "realised_vol",
            "atr",
            "vol_percentile",
            "target_return",
        ]
    ].dropna().reset_index(drop=True)

    rolling_vol_median = dataset.groupby("symbol")["realised_vol"].transform(
        lambda values: values.rolling(window=20, min_periods=5).median()
    )
    dataset["vol_regime"] = np.where(
        dataset["realised_vol"] >= rolling_vol_median.fillna(dataset["realised_vol"].median()),
        "high_vol",
        "low_vol",
    )
    dataset["trend_regime"] = np.where(dataset["log_return"] >= 0.0, "uptrend", "downtrend")
    dataset["regime"] = dataset["vol_regime"] + "|" + dataset["trend_regime"]
    return dataset


research_df = build_research_dataset(
    bars,
    return_horizon=config.return_horizon,
    volatility_window=config.volatility_window,
    atr_window=config.atr_window,
    volume_window=config.volume_window,
)

feature_columns = ["bar_portion", "log_return", "realised_vol", "atr", "vol_percentile"]
X = research_df[feature_columns].copy()
y = research_df["target_return"].rename("excess_return").copy()

X_pkg, y_pkg = build_feature_dataset(
    bars,
    return_horizon=config.return_horizon,
    volatility_window=config.volatility_window,
    atr_window=config.atr_window,
    volume_window=config.volume_window,
)

assert X.shape == X_pkg.shape
assert y.shape == y_pkg.shape
assert np.allclose(X.to_numpy(), X_pkg.to_numpy())
assert np.allclose(y.to_numpy(), y_pkg.to_numpy())

display(research_df.head())
display(X.describe().T)


## 5. Reproducibility snapshot

Every run should record what was executed. This cell makes the active configuration explicit before any model fitting begins.


In [ ]:
def package_version(name: str) -> str:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "not-installed"


run_snapshot = pd.DataFrame(
    {
        "run_timestamp_utc": [datetime.now(tz=UTC)],
        "python": [sys.version.split()[0]],
        "numpy": [package_version("numpy")],
        "pandas": [package_version("pandas")],
        "ngboost": [package_version("ngboost")],
        "matplotlib": [package_version("matplotlib")],
        "data_source": [data_source],
        "random_seed": [config.random_seed],
        "rows_after_feature_build": [len(research_df)],
        "unique_timestamps": [research_df["timestamp"].nunique()],
    }
)

config_table = pd.DataFrame(
    {
        "parameter": list(asdict(config).keys()),
        "value": [str(value) for value in asdict(config).values()],
    }
)

display(run_snapshot)
display(config_table)


## 6. Walk-forward evaluation and helper functions

The evaluation now uses expanding-window walk-forward folds built on unique timestamps. That keeps the split chronological across all symbols and makes the results less dependent on one arbitrary validation segment.


In [ ]:
def make_walk_forward_folds(
    dataset: pd.DataFrame,
    *,
    min_train_periods: int,
    test_periods: int,
    step_periods: int,
    max_folds: int,
) -> list[tuple[int, np.ndarray, np.ndarray]]:
    unique_timestamps = np.array(sorted(dataset["timestamp"].unique()))
    folds: list[tuple[int, np.ndarray, np.ndarray]] = []
    fold_number = 0

    train_end_idx = min_train_periods
    while train_end_idx + test_periods <= len(unique_timestamps) and fold_number < max_folds:
        train_timestamps = unique_timestamps[:train_end_idx]
        test_timestamps = unique_timestamps[train_end_idx : train_end_idx + test_periods]
        train_mask = dataset["timestamp"].isin(train_timestamps).to_numpy()
        test_mask = dataset["timestamp"].isin(test_timestamps).to_numpy()
        folds.append((fold_number, train_mask, test_mask))
        train_end_idx += step_periods
        fold_number += 1

    return folds


def evaluate_returns(returns: np.ndarray, positions: np.ndarray | None = None) -> dict[str, float]:
    values = np.asarray(returns, dtype=float).reshape(-1)
    if positions is None:
        positions = np.zeros_like(values)
    positions = np.asarray(positions, dtype=float).reshape(-1)
    turnover = float(np.mean(np.abs(np.diff(np.r_[0.0, positions])))) if positions.size else 0.0
    hit_rate = float(np.mean(values > 0.0)) if values.size else 0.0
    return {
        "mean_return": float(np.mean(values)) if values.size else 0.0,
        "volatility": float(np.std(values, ddof=0)) if values.size else 0.0,
        "sharpe": sharpe_ratio(values),
        "sortino": sortino_ratio(values),
        "adjusted_sharpe": adjusted_sharpe_ratio(values),
        "max_drawdown": max_drawdown(values),
        "turnover": turnover,
        "hit_rate": hit_rate,
        "avg_abs_position": float(np.mean(np.abs(positions))) if positions.size else 0.0,
    }


def bootstrap_metric_ci(
    returns: np.ndarray,
    metric_fn,
    *,
    n_boot: int = 300,
    seed: int = 0,
) -> tuple[float, float]:
    values = np.asarray(returns, dtype=float).reshape(-1)
    if values.size < 5:
        return (np.nan, np.nan)

    local_rng = np.random.default_rng(seed)
    estimates = []
    for _ in range(n_boot):
        sample = local_rng.choice(values, size=values.size, replace=True)
        estimates.append(metric_fn(sample))
    return (float(np.quantile(estimates, 0.05)), float(np.quantile(estimates, 0.95)))


def fit_linear_baseline(X_train: pd.DataFrame, y_train: pd.Series) -> np.ndarray:
    design = np.c_[np.ones(len(X_train)), X_train.to_numpy()]
    target = y_train.to_numpy()
    coef, _, _, _ = np.linalg.lstsq(design, target, rcond=None)
    return coef


def predict_linear_baseline(X_frame: pd.DataFrame, coefficients: np.ndarray) -> np.ndarray:
    design = np.c_[np.ones(len(X_frame)), X_frame.to_numpy()]
    return design @ coefficients


def positions_from_point_forecast(predictions: np.ndarray, *, allow_short: bool, cap: float = 2.0) -> np.ndarray:
    predictions = np.asarray(predictions, dtype=float).reshape(-1)
    scale = float(np.std(predictions, ddof=0))
    if not np.isfinite(scale) or np.isclose(scale, 0.0):
        scale = 1.0
    raw = np.tanh(predictions / scale) * cap
    if allow_short:
        return raw
    return np.clip(raw, 0.0, cap)


def distribution_nll(dist_obj: object, realized: np.ndarray) -> float:
    values = np.asarray(realized, dtype=float).reshape(-1)
    if values.size == 0:
        return np.nan
    try:
        logpdf = np.asarray(dist_obj.logpdf(values), dtype=float).reshape(-1)
        if logpdf.size != values.size:
            return np.nan
        return float(-np.mean(logpdf))
    except Exception:
        return np.nan


def interval_coverage(dist_obj: object, realized: np.ndarray, *, alpha: float = 0.1, n_samples: int = 1000) -> float:
    values = np.asarray(realized, dtype=float).reshape(-1)
    if values.size == 0:
        return np.nan
    try:
        lower = np.asarray(dist_obj.ppf(alpha / 2.0), dtype=float).reshape(-1)
        upper = np.asarray(dist_obj.ppf(1.0 - alpha / 2.0), dtype=float).reshape(-1)
    except Exception:
        sampled = np.asarray(dist_obj.sample(n_samples), dtype=float)
        if sampled.ndim == 1:
            sampled = sampled.reshape(n_samples, 1)
        lower = np.quantile(sampled, alpha / 2.0, axis=0)
        upper = np.quantile(sampled, 1.0 - alpha / 2.0, axis=0)
    if lower.size != values.size or upper.size != values.size:
        return np.nan
    return float(np.mean((values >= lower) & (values <= upper)))


def summarize_regime_performance(detail_df: pd.DataFrame) -> pd.DataFrame:
    return (
        detail_df.groupby(["strategy_name", "regime"])["strategy_return"]
        .agg(mean_return="mean", sharpe=sharpe_ratio, max_drawdown=max_drawdown, observations="size")
        .reset_index()
        .sort_values(["strategy_name", "regime"])
    )


folds = make_walk_forward_folds(
    research_df,
    min_train_periods=config.min_train_periods,
    test_periods=config.test_periods,
    step_periods=config.step_periods,
    max_folds=config.max_folds,
)

pd.DataFrame(
    [
        {
            "fold": fold_id,
            "train_rows": int(train_mask.sum()),
            "test_rows": int(test_mask.sum()),
            "train_end": research_df.loc[train_mask, "timestamp"].max(),
            "test_start": research_df.loc[test_mask, "timestamp"].min(),
            "test_end": research_df.loc[test_mask, "timestamp"].max(),
        }
        for fold_id, train_mask, test_mask in folds
    ]
)


## 7. Main experiment

Each fold trains the distributional model on the expanding training window and evaluates:
- fixed exposures: `0x`, `1x`, `2x`
- a simple linear point-forecast baseline
- distributional position sizing with `Normal` and `StudentT`

The detail table stores per-row outputs for later diagnostics, regime analysis, and failure inspection.


In [ ]:
result_rows: list[dict] = []
detail_frames: list[pd.DataFrame] = []
skipped_models: list[dict] = []

for fold_id, train_mask, test_mask in folds:
    train_df = research_df.loc[train_mask].reset_index(drop=True)
    test_df = research_df.loc[test_mask].reset_index(drop=True)

    X_train = train_df[feature_columns]
    y_train = train_df["target_return"]
    X_test = test_df[feature_columns]
    y_test = test_df["target_return"].to_numpy()

    baseline_coefficients = fit_linear_baseline(X_train, y_train)
    point_predictions = predict_linear_baseline(X_test, baseline_coefficients)
    point_positions = positions_from_point_forecast(point_predictions, allow_short=config.allow_short)

    baseline_position_map = {f"{exposure:.1f}x_static": np.full(len(X_test), exposure) for exposure in config.baseline_exposures}
    baseline_position_map["point_forecast"] = point_positions

    for strategy_name, baseline_positions in baseline_position_map.items():
        baseline_returns = baseline_positions * y_test
        metrics = evaluate_returns(baseline_returns, baseline_positions)
        sharpe_ci = bootstrap_metric_ci(baseline_returns, sharpe_ratio, seed=config.random_seed + fold_id)
        result_rows.append(
            {
                "fold": fold_id,
                "model_name": strategy_name,
                "strategy_family": "baseline",
                "transaction_cost": 0.0,
                "leverage_penalty": 0.0,
                "distribution_nll": np.nan,
                "interval_90_coverage": np.nan,
                "sharpe_ci_low": sharpe_ci[0],
                "sharpe_ci_high": sharpe_ci[1],
                **metrics,
            }
        )

        detail_frames.append(
            test_df.assign(
                fold=fold_id,
                strategy_name=strategy_name,
                strategy_family="baseline",
                predicted_mean=np.nan if strategy_name != "point_forecast" else point_predictions,
                predicted_scale=np.nan,
                position=baseline_positions,
                strategy_return=baseline_returns,
            )
        )

    for dist_name, n_estimators, learning_rate in config.model_grid:
        simulation_params = config.simulation_params(transaction_cost=0.0005, leverage_penalty=0.01)
        try:
            strategy = DistributionalStrategy(
                model_params={
                    "dist_name": dist_name,
                    "n_estimators": n_estimators,
                    "learning_rate": learning_rate,
                },
                simulation_params=simulation_params,
            )
            strategy.fit(X_train, y_train)
            positions = strategy.predict_positions(X_test)
            strategy_returns = positions * y_test

            dist_obj = strategy.model.predict_dist(X_test)
            try:
                predicted_mean = np.asarray(dist_obj.mean(), dtype=float).reshape(-1)
            except Exception:
                predicted_mean = np.full(len(X_test), np.nan)
            try:
                predicted_scale = np.asarray(dist_obj.scale, dtype=float).reshape(-1)
            except Exception:
                predicted_scale = np.full(len(X_test), np.nan)

            metrics = evaluate_returns(strategy_returns, positions)
            sharpe_ci = bootstrap_metric_ci(strategy_returns, sharpe_ratio, seed=config.random_seed + 100 + fold_id)
            result_rows.append(
                {
                    "fold": fold_id,
                    "model_name": f"distributional_{dist_name}",
                    "strategy_family": "distributional",
                    "distribution": dist_name,
                    "n_estimators": n_estimators,
                    "learning_rate": learning_rate,
                    "transaction_cost": simulation_params["transaction_cost"],
                    "leverage_penalty": simulation_params["leverage_penalty"],
                    "distribution_nll": distribution_nll(dist_obj, y_test),
                    "interval_90_coverage": interval_coverage(dist_obj, y_test),
                    "sharpe_ci_low": sharpe_ci[0],
                    "sharpe_ci_high": sharpe_ci[1],
                    **metrics,
                }
            )

            detail_frames.append(
                test_df.assign(
                    fold=fold_id,
                    strategy_name=f"distributional_{dist_name}",
                    strategy_family="distributional",
                    predicted_mean=predicted_mean,
                    predicted_scale=predicted_scale,
                    position=positions,
                    strategy_return=strategy_returns,
                )
            )
        except Exception as exc:
            skipped_models.append(
                {
                    "fold": fold_id,
                    "distribution": dist_name,
                    "n_estimators": n_estimators,
                    "learning_rate": learning_rate,
                    "error": str(exc),
                }
            )

results_df = pd.DataFrame(result_rows)
detail_df = pd.concat(detail_frames, ignore_index=True)

pooled_summary = (
    results_df.groupby(["model_name", "strategy_family"], dropna=False)
    .agg(
        folds=("fold", "nunique"),
        mean_return=("mean_return", "mean"),
        sharpe=("sharpe", "mean"),
        sortino=("sortino", "mean"),
        adjusted_sharpe=("adjusted_sharpe", "mean"),
        max_drawdown=("max_drawdown", "mean"),
        turnover=("turnover", "mean"),
        hit_rate=("hit_rate", "mean"),
        avg_abs_position=("avg_abs_position", "mean"),
        distribution_nll=("distribution_nll", "mean"),
        interval_90_coverage=("interval_90_coverage", "mean"),
    )
    .sort_values("adjusted_sharpe", ascending=False)
    .reset_index()
)

display(results_df.sort_values(["fold", "adjusted_sharpe"], ascending=[True, False]))
display(pooled_summary)
if skipped_models:
    display(pd.DataFrame(skipped_models))


## 8. Diagnostics and interpretation

The next cells make the strategy behavior easier to inspect:
- cumulative return curves
- position distribution
- position vs realized return scatter
- regime summaries
- worst loss events for the top distributional model


In [ ]:
best_distributional_name = (
    pooled_summary.loc[pooled_summary["strategy_family"] == "distributional", "model_name"].iloc[0]
    if (pooled_summary["strategy_family"] == "distributional").any()
    else None
)

plot_candidates = ["1.0x_static", "point_forecast"]
if best_distributional_name is not None:
    plot_candidates.append(best_distributional_name)

chart_df = detail_df[detail_df["strategy_name"].isin(plot_candidates)].copy()
chart_df = chart_df.sort_values(["strategy_name", "timestamp"])
chart_df["cumulative_equity"] = chart_df.groupby("strategy_name")["strategy_return"].transform(lambda values: np.cumprod(1.0 + values))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for strategy_name, frame in chart_df.groupby("strategy_name"):
    axes[0, 0].plot(frame["timestamp"], frame["cumulative_equity"], label=strategy_name)
axes[0, 0].set_title("Cumulative equity by strategy")
axes[0, 0].legend()

if best_distributional_name is not None:
    best_frame = detail_df[detail_df["strategy_name"] == best_distributional_name].copy()
    axes[0, 1].hist(best_frame["position"], bins=12, edgecolor="black")
    axes[0, 1].set_title(f"Position distribution: {best_distributional_name}")

    axes[1, 0].scatter(best_frame["position"], best_frame["target_return"], alpha=0.5)
    axes[1, 0].set_title("Position vs realized forward return")
    axes[1, 0].set_xlabel("Position")
    axes[1, 0].set_ylabel("Forward return")

    top_examples = best_frame[["timestamp", "symbol", "predicted_mean", "predicted_scale", "target_return"]].dropna().head(80)
    axes[1, 1].plot(top_examples["timestamp"], top_examples["predicted_mean"], label="predicted mean")
    axes[1, 1].fill_between(
        top_examples["timestamp"],
        top_examples["predicted_mean"] - 1.64 * top_examples["predicted_scale"],
        top_examples["predicted_mean"] + 1.64 * top_examples["predicted_scale"],
        alpha=0.25,
        label="approx. 90% interval",
    )
    axes[1, 1].plot(top_examples["timestamp"], top_examples["target_return"], label="realized return", linewidth=1.2)
    axes[1, 1].set_title("Predicted mean and uncertainty vs realized return")
    axes[1, 1].legend()
else:
    axes[0, 1].axis("off")
    axes[1, 0].axis("off")
    axes[1, 1].axis("off")

plt.tight_layout()
plt.show()

regime_summary = summarize_regime_performance(detail_df)
display(regime_summary)

if best_distributional_name is not None:
    failure_cases = (
        detail_df[detail_df["strategy_name"] == best_distributional_name]
        .sort_values("strategy_return")
        .loc[:, ["timestamp", "symbol", "position", "target_return", "strategy_return", "regime", "predicted_mean", "predicted_scale"]]
        .head(10)
    )
    display(failure_cases)


## 9. Sensitivity analysis

This ablation keeps the best distribution family fixed and varies transaction cost and leverage penalty. The goal is to check whether the headline result survives small realism changes rather than collapsing immediately.


In [ ]:
sensitivity_rows: list[dict] = []

best_dist = None
if best_distributional_name is not None:
    best_dist = best_distributional_name.split("_", 1)[1]

if best_dist is not None:
    for transaction_cost in config.cost_grid:
        for leverage_penalty in config.leverage_penalty_grid:
            fold_returns = []
            fold_positions = []

            for fold_id, train_mask, test_mask in folds:
                train_df = research_df.loc[train_mask].reset_index(drop=True)
                test_df = research_df.loc[test_mask].reset_index(drop=True)
                X_train = train_df[feature_columns]
                y_train = train_df["target_return"]
                X_test = test_df[feature_columns]
                y_test = test_df["target_return"].to_numpy()

                try:
                    strategy = DistributionalStrategy(
                        model_params={
                            "dist_name": best_dist,
                            "n_estimators": 20,
                            "learning_rate": 0.05,
                        },
                        simulation_params=config.simulation_params(
                            transaction_cost=transaction_cost,
                            leverage_penalty=leverage_penalty,
                        ),
                    )
                    strategy.fit(X_train, y_train)
                    positions = strategy.predict_positions(X_test)
                    fold_positions.append(positions)
                    fold_returns.append(positions * y_test)
                except Exception:
                    fold_positions = []
                    fold_returns = []
                    break

            stacked_returns = np.concatenate(fold_returns) if fold_returns else np.array([])
            stacked_positions = np.concatenate(fold_positions) if fold_positions else np.array([])
            if stacked_returns.size:
                sensitivity_rows.append(
                    {
                        "distribution": best_dist,
                        "transaction_cost": transaction_cost,
                        "leverage_penalty": leverage_penalty,
                        **evaluate_returns(stacked_returns, stacked_positions),
                    }
                )

sensitivity_df = pd.DataFrame(sensitivity_rows)
if not sensitivity_df.empty:
    sensitivity_pivot = sensitivity_df.pivot(index="transaction_cost", columns="leverage_penalty", values="adjusted_sharpe")
    display(sensitivity_df.sort_values("adjusted_sharpe", ascending=False))
    display(sensitivity_pivot)

    ax = sensitivity_pivot.plot(kind="bar", figsize=(10, 5))
    ax.set_title(f"Adjusted Sharpe sensitivity for {best_dist}")
    ax.set_ylabel("Adjusted Sharpe")
    plt.tight_layout()
    plt.show()
else:
    print("Sensitivity analysis skipped because no distributional strategy completed.")


## 10. Conclusions and next experiments

This cell converts the tables above into a compact research summary. Treat it as a notebook-native experiment log: it should be easy to skim and easy to compare against future reruns.


In [ ]:
conclusion_lines = []

best_row = pooled_summary.iloc[0]
conclusion_lines.append(
    f"Best pooled strategy: {best_row['model_name']} with mean adjusted Sharpe {best_row['adjusted_sharpe']:.3f} and mean max drawdown {best_row['max_drawdown']:.3f}."
)

if best_distributional_name is not None:
    dist_row = pooled_summary.loc[pooled_summary["model_name"] == best_distributional_name].iloc[0]
    point_row = pooled_summary.loc[pooled_summary["model_name"] == "point_forecast"].iloc[0]
    static_row = pooled_summary.loc[pooled_summary["model_name"] == "1.0x_static"].iloc[0]
    conclusion_lines.append(
        f"Top distributional model: {best_distributional_name} with adjusted Sharpe {dist_row['adjusted_sharpe']:.3f}, versus point forecast {point_row['adjusted_sharpe']:.3f} and static 1x {static_row['adjusted_sharpe']:.3f}."
    )
    if pd.notna(dist_row["interval_90_coverage"]):
        conclusion_lines.append(
            f"Approximate 90% interval coverage for {best_distributional_name}: {dist_row['interval_90_coverage']:.2%}."
        )
    conclusion_lines.append(
        "Check the failure-case table before trusting the headline result. Large losses with high exposure are the fastest way this approach fails in practice."
    )

if not sensitivity_df.empty:
    sensitivity_best = sensitivity_df.sort_values("adjusted_sharpe", ascending=False).iloc[0]
    conclusion_lines.append(
        f"Best sensitivity setting: transaction_cost={sensitivity_best['transaction_cost']:.4f}, leverage_penalty={sensitivity_best['leverage_penalty']:.3f}, adjusted Sharpe={sensitivity_best['adjusted_sharpe']:.3f}."
    )

conclusion_lines.append(
    "Next experiments: add a stronger point-prediction baseline, widen the symbol universe, test long-short sizing, and move from fold means to formal significance tests on strategy differences."
)

print("\n".join(conclusion_lines))
